# LaughTuned — Demo Notebook

Fine-tuning Mistral-7B-Instruct-v0.2 for comedy writing using **DPO** and **KTO**, both implemented from scratch in PyTorch (CS 5788, Cornell).

This notebook demonstrates the pipeline. The main engine lives in the `.py` modules of the repo.

## Step 0 — Environment Setup

Bootstraps the Colab runtime: clones the code repo, installs dependencies, mounts Drive, sets seeds, and verifies the GPU.

In [ ]:
# === Colab bootstrap: clone repo, install deps, cd into the code dir ===
# Re-running this cell after a `git pull` is safe — it both refreshes the
# code on disk AND drops cached project modules from `sys.modules` so the
# next `import` reads the new file.
import os
import subprocess
import sys

REPO_URL = "https://github.com/pcatattacks/laughtuned.git"
REPO_DIR = "/content/laughtuned"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        print(f"Cloning {REPO_URL} into {REPO_DIR} ...")
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        print(f"Updating existing checkout at {REPO_DIR} ...")
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    os.chdir(REPO_DIR)
    print(f"cwd: {os.getcwd()}")
    print("Installing dependencies (quiet) ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True,
    )
else:
    print("Not on Colab — assuming dependencies are already installed and cwd is the repo root.")

# --- Module-cache reset ----------------------------------------------------
# Python caches imported modules in `sys.modules`. After `git pull` updates
# the .py files on disk, a re-`import` still returns the OLD code from the
# cache. Drop our project's modules so the next import reads the freshly
# pulled code. Safe on first run too (it's just a no-op when nothing is cached).
_PROJECT_TOP_LEVELS = ("config", "data", "models", "utils")
_dropped = []
for _mod_name in list(sys.modules):
    if _mod_name in _PROJECT_TOP_LEVELS or _mod_name.startswith(
        tuple(f"{p}." for p in _PROJECT_TOP_LEVELS)
    ):
        del sys.modules[_mod_name]
        _dropped.append(_mod_name)
if _dropped:
    print(f"Cleared module cache for {len(_dropped)} project modules.")
else:
    print("Module cache empty (fresh kernel).")

In [2]:
# === Mount Drive and create the artifact tree ===
from config import CONFIG
from utils.drive_utils import mount_drive, ensure_drive_dirs

mount_drive()
ensure_drive_dirs(CONFIG)

In [3]:
# === Set all random seeds for reproducibility ===
import random
import numpy as np
import torch

SEED = CONFIG["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"Seeded with {SEED}")

In [4]:
# === GPU sanity check ===
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name} | total VRAM: {total_gb:.1f} GB")
else:
    print("No GPU detected. Switch the Colab runtime to T4 or A100 before continuing.")

## Step 1 — Load the base model with QLoRA

`load_model_and_tokenizer` does the QLoRA setup: 4-bit NF4 with double quantization, then LoRA adapters (rank 16) on the attention projections. Expect a ~4 GB base load; trainable params should be well under 1% of total.

In [5]:
from models.load_model import load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer(CONFIG)

## `compute_log_probs` — the shared primitive for DPO and KTO

Both losses score a response by the total log-probability the policy assigns to it given the prompt: `log π(y | x) = Σ_t log π(y_t | x, y_<t)`. The implementation in [`models/log_probs.py`](models/log_probs.py) handles three subtleties:

1. **Shift by one.** A causal LM's output at position `t` predicts the token at `t+1`, so we align `logits[:, :-1]` with `input_ids[:, 1:]` and `label_mask[:, 1:]`.
2. **Prompt tokens contribute zero.** The label mask is 1 only on response tokens (and 0 on prompt tokens *and* padding); after shifting we multiply by it before summing.
3. **Sum, not mean.** DPO and KTO are derived from the total sequence log-probability; length-normalization changes the optimization landscape.

The smoke test below verifies shape, sign, and masking on a synthetic batch.

In [6]:
# === Smoke test for compute_log_probs ===
from models.log_probs import compute_log_probs

# Synthetic batch: 2 examples, 8 tokens each.
# First 4 tokens are "prompt" (mask=0), last 4 are "response" (mask=1).
B, T = 2, 8
device = next(model.parameters()).device
vocab_size = model.config.vocab_size

input_ids_B_T = torch.randint(0, vocab_size, (B, T), device=device)
attention_mask_B_T = torch.ones(B, T, dtype=torch.long, device=device)
label_mask_B_T = torch.zeros(B, T, dtype=torch.long, device=device)
label_mask_B_T[:, T // 2 :] = 1  # response = second half

with torch.no_grad():
    log_probs_B = compute_log_probs(
        model, input_ids_B_T, attention_mask_B_T, label_mask_B_T
    )

# Test 1: shape
assert log_probs_B.shape == (B,), f"expected ({B},), got {tuple(log_probs_B.shape)}"

# Test 2: all values <= 0
assert (log_probs_B <= 0).all(), f"log-probs should be non-positive, got {log_probs_B}"

# Test 3: zeroed mask -> zero output
zero_mask_B_T = torch.zeros_like(label_mask_B_T)
with torch.no_grad():
    zero_log_probs_B = compute_log_probs(
        model, input_ids_B_T, attention_mask_B_T, zero_mask_B_T
    )
assert torch.allclose(
    zero_log_probs_B, torch.zeros_like(zero_log_probs_B)
), f"expected all-zero output for empty mask, got {zero_log_probs_B}"

print("compute_log_probs passed all 3 smoke tests.")
print(f"sample log-probs: {log_probs_B.tolist()}")

## Step 2 — Article ingestion with historical backstories

We sample articles from eight Guardian sections (politics, business, technology, sport, culture, science, environment, world). For each, Claude synthesizes a 150-word backstory drawn from the most relevant older Guardian articles on the same topic. Every article ends up with **two** context variants:

- `short_context` — headline + first 400 tokens of today's body
- `long_context` — synthesized backstory + the short context

Holding the comedy-prompt template constant and varying only the context lets us isolate whether richer historical grounding produces better jokes.

Set `GUARDIAN_API_KEY` and `ANTHROPIC_API_KEY` in Colab secrets (left sidebar → 🔑 Secrets) before running the next cell.

In [ ]:
# === Load API keys from Colab secrets (or prompt as fallback) ===
def _load_secret(name: str, required: bool = True) -> str:
    """Try Colab secrets first; fall back to getpass only if required."""
    try:
        from google.colab import userdata  # type: ignore[import-not-found]
        return userdata.get(name)
    except Exception:
        if not required:
            return ""
        import getpass
        return getpass.getpass(f"Paste {name}: ")

CONFIG["guardian_api_key"]   = _load_secret("GUARDIAN_API_KEY")
CONFIG["guardian_api_key_2"] = _load_secret("GUARDIAN_API_KEY_2", required=False)
CONFIG["anthropic_api_key"]  = _load_secret("ANTHROPIC_API_KEY")

n_guardian = sum(1 for k in (CONFIG["guardian_api_key"], CONFIG["guardian_api_key_2"]) if k)
print(f"API keys loaded. Guardian keys: {n_guardian} | Anthropic: {'yes' if CONFIG['anthropic_api_key'] else 'no'}")

In [ ]:
# === Smoke test: one round-trip through Guardian + Claude before the full run ===
# Exercises search_section -> search_related -> synthesize_backstory on a single
# article and prints the assembled record. Does not write to disk; safe to run
# before cell 12. Costs ~3 Guardian calls + 1 Claude call (~$0.01).
from datetime import datetime, timedelta, timezone
import anthropic
from data.fetch_articles import (
    search_section,
    search_related,
    synthesize_backstory,
    extract_short_context,
    extract_topic_query,
    BACKSTORY_LOOKBACK_DAYS,
    BACKSTORY_BUFFER_DAYS,
)

guardian_keys = [
    k for k in (CONFIG["guardian_api_key"], CONFIG["guardian_api_key_2"]) if k
]
claude = anthropic.Anthropic(api_key=CONFIG["anthropic_api_key"])
office = CONFIG.get("guardian_production_office")
assert guardian_keys, "no Guardian keys loaded"
assert CONFIG["anthropic_api_key"], "no Anthropic key loaded"

# 1. Pull a few recent politics articles, take the first one
today = datetime.now(timezone.utc).date()
section_articles = search_section(
    guardian_keys,
    section="politics",
    from_date=(today - timedelta(days=30)).isoformat(),
    to_date=today.isoformat(),
    page_size=3,
    production_office=office,
)
assert section_articles, "section search returned no articles"
raw = section_articles[0]
headline = (raw.get("fields") or {}).get("headline") or raw.get("webTitle", "")
print(f"Article: {headline!r}")
print(f"  published: {raw.get('webPublicationDate')}")

# 2. Build short context + topic query
short_context = extract_short_context(raw, tokenizer)
topic_query = extract_topic_query(raw)
print(f"  short_context: {len(short_context)} chars")
print(f"  topic_query:   {topic_query[:80]}")

# 3. Related search (top 3 by relevance within the lookback window)
published_date = datetime.fromisoformat(
    raw["webPublicationDate"].replace("Z", "+00:00")
).date()
older = search_related(
    guardian_keys,
    topic_query=topic_query,
    from_date=(published_date - timedelta(days=BACKSTORY_LOOKBACK_DAYS)).isoformat(),
    to_date=(published_date - timedelta(days=BACKSTORY_BUFFER_DAYS)).isoformat(),
    production_office=office,
)
older = [o for o in older if o["id"] != raw["id"]]
print(f"\nHistorical articles ({len(older)}):")
for o in older:
    o_head = (o.get("fields") or {}).get("headline") or o.get("webTitle", "")
    print(f"  - {o.get('webPublicationDate', '?')[:10]}  {o_head}")

# 4. Backstory synthesis via Claude
backstory = synthesize_backstory(claude, CONFIG["judge_model"], raw, older, tokenizer)
print(f"\nBackstory ({len(backstory)} chars):")
print(backstory[:400] + ("..." if len(backstory) > 400 else ""))

# 5. Final long_context shape
long_context = f"{backstory}\n\n{short_context}" if backstory else short_context
print(
    f"\nlong_context: {len(long_context)} chars "
    f"(+{len(long_context) - len(short_context)} vs short_context)"
)
print("\nSmoke test passed.")

In [ ]:
# === Ingest articles (idempotent: resumes from disk if interrupted) ===
from data.fetch_articles import ingest_articles

articles = ingest_articles(CONFIG, tokenizer)

print(f"\nTotal articles: {len(articles)}")
print(f"Sections covered: {sorted({a['section'] for a in articles})}")

sample = articles[0]
print(f"\n--- Sample article ---")
print(f"headline:      {sample['headline']}")
print(f"section:       {sample['section']}")
print(f"published:     {sample['published_at']}")
print(f"older refs:    {len(sample['older_article_ids'])} historical articles used")
print(f"short_context: {len(sample['short_context'])} chars")
print(f"long_context:  {len(sample['long_context'])} chars  "
      f"(+{len(sample['long_context']) - len(sample['short_context'])} from backstory)")
print(f"\nshort_context preview:\n{sample['short_context'][:300]}...")
print(f"\nlong_context preview:\n{sample['long_context'][:400]}...")

## Step 3 — Build comedy prompts

For each ingested article, we assign a random comedy style (observational, absurdist, one-liner) and generate **two** prompt variants using the same template:

- `prompt_text_short` — `<context>` contains just today's article.
- `prompt_text_long` — `<context>` contains `<background>{backstory}</background><latest>{today}</latest>`.

The template wording is byte-identical between the two versions; only the contents of `<context>` differ. This isolates context length as the only experimental variable.

Prompts are split before any generation:
- **30 eval prompts** held out completely from training
- Remaining 90/10 → train / val

In [ ]:
# === Build prompts from the ingested articles ===
from data.fetch_articles import load_existing_articles
from data.build_prompts import build_prompts

prompts = build_prompts(CONFIG, load_existing_articles(CONFIG))

# Sanity-print one short and one long prompt from the same record so the
# template alignment is visible.
sample = next(p for p in prompts if p["prompt_text_long"] != p["prompt_text_short"])
print(f"\n--- Sample prompt (style={sample['style']}, split={sample['split']}) ---")
print(f"\nprompt_text_short ({len(sample['prompt_text_short'])} chars):\n")
print(sample["prompt_text_short"][:500] + "...")
print(f"\nprompt_text_long ({len(sample['prompt_text_long'])} chars):\n")
print(sample["prompt_text_long"][:600] + "...")

## Step 4 — Generate response pairs from the base Mistral model

For every train/val prompt we sample **2 responses** (response_a, response_b) from base Mistral at both context lengths. Independent sampling on duplicated prompts in the same forward pass produces two different completions. We also sample **1 baseline** per eval prompt for later cross-judge comparison.

Generation hyperparameters per spec: `temperature=0.9, top_p=0.95, max_new_tokens=200`. Outputs persist to disk after every batch (resume-friendly). Expect ~3–8 hours total on T4 (faster on A100); pairs and baselines are independent so you can run them in any order.

In [ ]:
# === Generate response pairs for train/val prompts (both context lengths) ===
from data.generate_pairs import generate_pairs, generate_baselines

# Short context first (smaller prompts, easier to debug if something OOMs)
pairs_short = generate_pairs(model, tokenizer, CONFIG, prompts, context_length="short", batch_size=4)
pairs_long  = generate_pairs(model, tokenizer, CONFIG, prompts, context_length="long",  batch_size=4)

# Eval baselines (1 response per held-out eval prompt)
baselines_short = generate_baselines(model, tokenizer, CONFIG, prompts, context_length="short", batch_size=8)
baselines_long  = generate_baselines(model, tokenizer, CONFIG, prompts, context_length="long",  batch_size=8)

print(f"\nTotal generations on disk: pairs={len(pairs_short) + len(pairs_long)} new, "
      f"baselines={len(baselines_short) + len(baselines_long)} new")

## Step 5 — LLM judge with structured rubric

Claude Sonnet scores each generated pair on three dimensions — context engagement, comedic technique, and surprise — each from 1 to 5, then picks the funnier overall. The composite (mean) score drives both:

- **DPO** preference pairs: winner = chosen, loser = rejected.
- **KTO** binary labels: composite > `kto_desirable_threshold` → desirable, composite < `kto_undesirable_threshold` → undesirable, else dropped.

Same rubric, two consumers — keeps the evaluation criterion consistent across algorithms.

In [ ]:
# === Judge every generated pair on the rubric ===
from data.generate_pairs import _load_jsonl, _generations_path
from data.judge import judge_all_pairs, build_contexts_map

# Load the generation pairs from Step 4
generations = _load_jsonl(_generations_path(CONFIG))
contexts_by_prompt = build_contexts_map(prompts)

# Cost preview (so you can ctrl-stop before commit if needed)
n_to_judge = sum(
    1 for g in generations
    if g.get("prompt_id") and g.get("context_length")
)
print(f"About to judge {n_to_judge} pairs at ~$0.01 each = ~${n_to_judge * 0.01:.2f}")

rubric_records = judge_all_pairs(CONFIG, generations, contexts_by_prompt)

## Step 6 — Build tokenized DPO / KTO datasets

The same rubric judgments feed both algorithms, just consumed differently:

- **DPO** examples are `(prompt, chosen, rejected)`, where the chosen response is the rubric winner.
- **KTO** examples are `(prompt, response, label)`, where each response with a non-ambiguous composite score becomes one example.

Each example is pre-tokenized into `(input_ids, attention_mask, label_mask)` at construction time so the training loop is just tensor indexing. Splits are inherited from Step 3.

In [ ]:
# === Build DPO/KTO example lists, save to disk, and make all 4 sets of DataLoaders ===
from data.judge import load_existing_rubric_records
from data.prepare_datasets import (
    build_dpo_examples,
    build_kto_examples,
    save_examples,
    make_dpo_loaders,
    make_kto_loaders,
)

rubric_records = load_existing_rubric_records(CONFIG)
dpo_examples = build_dpo_examples(rubric_records, prompts)
kto_examples = build_kto_examples(rubric_records, prompts)
save_examples(CONFIG, dpo_examples, kto_examples)

# Build loaders for every (algorithm × context_length) combination. We'll
# train 4 variants total, sharing the same example pool across contexts.
loaders = {}
for ctx in ("short", "long"):
    dpo_train, dpo_val = make_dpo_loaders(
        dpo_examples, tokenizer,
        max_length=CONFIG["max_seq_length"],
        batch_size=CONFIG["batch_size"],
        context_length=ctx,
    )
    kto_train, kto_val = make_kto_loaders(
        kto_examples, tokenizer,
        max_length=CONFIG["max_seq_length"],
        batch_size=CONFIG["batch_size"],
        context_length=ctx,
    )
    loaders[("dpo", ctx)] = (dpo_train, dpo_val)
    loaders[("kto", ctx)] = (kto_train, kto_val)

# Summary
for (algo, ctx), (tl, vl) in loaders.items():
    n_train = len(tl.dataset)
    n_val = len(vl.dataset)
    extra = ""
    if algo == "kto":
        n_des = sum(
            1 for e in kto_examples
            if e["split"] == "train" and e["context_length"] == ctx and e["label"] == 1.0
        )
        n_und = sum(
            1 for e in kto_examples
            if e["split"] == "train" and e["context_length"] == ctx and e["label"] == 0.0
        )
        extra = f" ({n_des} desirable + {n_und} undesirable train)"
    print(f"{algo.upper()} {ctx}: {n_train} train, {n_val} val{extra}")

## Step 7 — Precompute reference log-probabilities

DPO and KTO both score the policy's responses against a *frozen* reference. By computing those reference log-probs once before training and caching them to disk, we keep only one model in VRAM during the training loop and look up reference scores by example index.

We use the same PEFT-wrapped model with the LoRA adapter **disabled** (`model.disable_adapter()`) — which is mathematically equivalent to the base model since fresh LoRA initializes the B matrix to zero.

In [ ]:
# === Precompute reference log-probs for all 4 variants ===
# Idempotent: each precompute_* checks for an existing cached tensor on Drive
# and returns it without re-computing. So re-running this cell is cheap.
from models.ref_log_probs import (
    precompute_dpo_ref_logps,
    precompute_kto_ref_logps,
)

ref_logps = {}
for ctx in ("short", "long"):
    dpo_train, dpo_val = loaders[("dpo", ctx)]
    kto_train, kto_val = loaders[("kto", ctx)]

    ref_logps[("dpo", ctx, "train")] = precompute_dpo_ref_logps(
        model, dpo_train, CONFIG, split="train", context_length=ctx,
    )
    ref_logps[("dpo", ctx, "val")] = precompute_dpo_ref_logps(
        model, dpo_val, CONFIG, split="val", context_length=ctx,
    )
    ref_logps[("kto", ctx, "train")] = precompute_kto_ref_logps(
        model, kto_train, CONFIG, split="train", context_length=ctx,
    )
    ref_logps[("kto", ctx, "val")] = precompute_kto_ref_logps(
        model, kto_val, CONFIG, split="val", context_length=ctx,
    )

for (algo, ctx, split), tensor_or_dict in ref_logps.items():
    if algo == "dpo":
        shape = tensor_or_dict["chosen"].shape
    else:
        shape = tensor_or_dict.shape
    print(f"  {algo} {ctx} {split}: {tuple(shape)}")

## Step 8 — DPO and KTO loss functions

Both losses live in [`models/dpo.py`](models/dpo.py) and [`models/kto.py`](models/kto.py). They consume per-example log-probs (under both the current policy and the frozen reference) and produce a scalar loss plus a metrics dict.

The synthetic smoke tests below confirm directional correctness on small fabricated batches:

- **DPO**: loss should drop when the policy assigns more probability mass to chosen vs rejected.
- **KTO**: loss should drop when desirable responses have higher log-ratios than undesirable ones.

In [ ]:
# === DPO loss smoke test ===
from models.dpo import dpo_loss

torch.manual_seed(0)
B = 8
ref_chosen = torch.randn(B)
ref_rejected = torch.randn(B)

# Policy A: chosen far above rejected (should give low loss)
pol_chosen_good = ref_chosen + 1.0
pol_rejected_bad = ref_rejected - 1.0
loss_good, _ = dpo_loss(
    pol_chosen_good, pol_rejected_bad, ref_chosen, ref_rejected, beta=0.1
)

# Policy B: chosen below rejected (should give higher loss)
pol_chosen_bad = ref_chosen - 1.0
pol_rejected_good = ref_rejected + 1.0
loss_bad, metrics = dpo_loss(
    pol_chosen_bad, pol_rejected_good, ref_chosen, ref_rejected, beta=0.1
)

assert loss_good.item() < loss_bad.item(), (
    f"DPO loss should be lower when policy prefers chosen: "
    f"good={loss_good.item():.4f}, bad={loss_bad.item():.4f}"
)
assert metrics["accuracy"] <= 0.5, "accuracy on 'bad' should be <= 50%"
for key in ("loss", "reward_margin", "accuracy", "chosen_rewards_mean", "rejected_rewards_mean"):
    assert key in metrics, f"metrics dict missing key: {key}"
print(f"DPO smoke test passed | good loss = {loss_good.item():.4f}, bad loss = {loss_bad.item():.4f}")

In [ ]:
# === KTO loss smoke test ===
from models.kto import kto_loss

torch.manual_seed(0)
B = 8
labels = torch.tensor([1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0])
ref_logps = torch.randn(B)

# Policy A: desirables above reference, undesirables below (should give low loss)
pol_logps_good = ref_logps + torch.where(labels == 1.0, 1.0, -1.0)
loss_good, _ = kto_loss(pol_logps_good, ref_logps, labels, beta=0.1)

# Policy B: the opposite (should give higher loss)
pol_logps_bad = ref_logps + torch.where(labels == 1.0, -1.0, 1.0)
loss_bad, metrics = kto_loss(pol_logps_bad, ref_logps, labels, beta=0.1)

assert loss_good.item() < loss_bad.item(), (
    f"KTO loss should be lower when desirables outscore reference: "
    f"good={loss_good.item():.4f}, bad={loss_bad.item():.4f}"
)
for key in (
    "loss", "z_ref", "v_desirable_mean", "v_undesirable_mean",
    "log_ratio_mean", "lambda_desirable", "lambda_undesirable",
):
    assert key in metrics, f"metrics dict missing key: {key}"
print(f"KTO smoke test passed | good loss = {loss_good.item():.4f}, bad loss = {loss_bad.item():.4f}")

## Step 9 — Train DPO and KTO

The training loop in [`models/train.py`](models/train.py) is shared across both algorithms — `loss_fn` is the only thing that changes. It handles AdamW + cosine LR + warmup, gradient accumulation, gradient clipping, periodic eval, checkpointing, and early stopping. The logger streams metrics to TensorBoard + JSONL on Drive; the checkpointer manages LoRA adapter snapshots under `<drive_root>/checkpoints/<experiment_name>/`.

Expect ~1–3 hours per variant on T4 (~30 min on A100) depending on dataset size.

In [ ]:
# === Train all 4 variants: DPO × {short, long} + KTO × {short, long} ===
# Each variant gets a fresh LoRA adapter. Between variants we reload the
# base model so previous training doesn't leak into the next run.
import gc

from models.dpo import dpo_loss
from models.kto import kto_loss
from models.train import train
from models.load_model import load_model_and_tokenizer
from utils.logging import TrainingLogger
from utils.checkpointing import CheckpointManager

VARIANTS = [
    ("dpo", "short", dpo_loss),
    ("dpo", "long",  dpo_loss),
    ("kto", "short", kto_loss),
    ("kto", "long",  kto_loss),
]

summaries = {}
for i, (algo, ctx, loss_fn) in enumerate(VARIANTS):
    name = f"{algo}_rubric_{ctx}" if algo == "dpo" else f"{algo}_binary_{ctx}"
    print(f"\n{'=' * 70}\n[{i + 1}/{len(VARIANTS)}] Training {name}\n{'=' * 70}")

    train_loader, val_loader = loaders[(algo, ctx)]
    ref_train = ref_logps[(algo, ctx, "train")]
    ref_val = ref_logps[(algo, ctx, "val")]

    exp_logger = TrainingLogger(CONFIG, name)
    exp_ckpt = CheckpointManager(CONFIG, name)
    summaries[name] = train(
        model=model,
        train_dataloader=train_loader,
        val_dataloader=val_loader,
        ref_log_probs_train=ref_train,
        ref_log_probs_val=ref_val,
        loss_fn=loss_fn,
        config=CONFIG,
        experiment_name=name,
        logger=exp_logger,
        checkpointer=exp_ckpt,
    )
    exp_logger.close()

    # Save the best adapter for this variant under a stable name so we can
    # reload it during the Step 11 cross-judge. The checkpointer's `best/`
    # already contains it; we just keep a pointer.
    print(f"  best adapter: {exp_ckpt._best_path}")

    # Reload the base model before the next variant so the LoRA weights
    # are reset to their initial (B=0) state. Skip on the last iteration.
    if i < len(VARIANTS) - 1:
        del model
        gc.collect()
        torch.cuda.empty_cache()
        print("  reloading base model for next variant ...")
        model, tokenizer = load_model_and_tokenizer(CONFIG)

print("\n" + "=" * 70 + "\nAll trainings complete:")
for name, s in summaries.items():
    print(
        f"  {name:30s}  best_val_loss={s['best_val_loss']:.4f}  "
        f"@step {s['best_step']}  total_steps={s['total_steps']}"
    )

## Step 10 / 11 — Final eval: generate from each variant, then LLM cross-judge

For each of the 30 held-out eval prompts, we sample one response from:

- **base Mistral** (LoRA disabled) — on both short and long context prompts
- **dpo_rubric_short** — on short context only (it was trained on short)
- **dpo_rubric_long** — on long context only
- **kto_binary_short** — on short context only
- **kto_binary_long** — on long context only

Then Claude runs two 3-way rankings per prompt (base vs DPO vs KTO at each context length) to produce pairwise win rates. The cross-judge uses a different prompt template than the rubric judge used in training, to avoid circularity.

In [ ]:
# === Generate responses from base + each trained variant on the eval prompts ===
from eval.generate_eval import (
    generate_for_variant,
    load_final_generations,
    index_generations,
    VARIANT_NAMES,
)
from utils.checkpointing import CheckpointManager

# Base baseline: same model, LoRA disabled. Runs on BOTH short and long prompts.
for ctx in ("short", "long"):
    print(f"Generating base/{ctx} responses ...")
    with model.disable_adapter():
        generate_for_variant(
            model, tokenizer, CONFIG, prompts,
            variant_name="base",
            context_length=ctx,
            batch_size=8,
        )

# Trained variants: load each one's BEST adapter, generate on its matching context.
for variant in (
    "dpo_rubric_short", "dpo_rubric_long",
    "kto_binary_short", "kto_binary_long",
):
    ctx = "short" if variant.endswith("short") else "long"
    print(f"Loading {variant} adapter and generating on {ctx} prompts ...")
    CheckpointManager(CONFIG, variant).load_best(model)
    generate_for_variant(
        model, tokenizer, CONFIG, prompts,
        variant_name=variant,
        context_length=ctx,
        batch_size=8,
    )

final_gens = load_final_generations(CONFIG)
gens_by_prompt = index_generations(final_gens)
print(f"\nTotal final-eval generations on disk: {len(final_gens)}")
print(f"Prompts covered: {len(gens_by_prompt)}")

In [ ]:
# === Run cross-judge and compute pairwise win rates ===
from eval.judge_eval import (
    cross_judge_all,
    load_existing_cross_judge,
    compute_pairwise_winrates,
)

new_records = cross_judge_all(CONFIG, prompts, gens_by_prompt)
all_records = load_existing_cross_judge(CONFIG)
print(f"Total cross-judge records: {len(all_records)} ({len(new_records)} new)")

for ctx in ("short", "long"):
    print(f"\n=== Pairwise win rates — {ctx} context ===")
    rates = compute_pairwise_winrates(all_records, context_length=ctx)
    for (a, b), data in sorted(rates.items()):
        wr_a = data.get(f"win_rate_{a}", 0.0)
        wins_a = int(data.get(f"{a}_wins", 0))
        wins_b = int(data.get(f"{b}_wins", 0))
        print(f"  {a:30s} vs {b:30s}: {a} wins {wins_a}/{wins_a + wins_b} ({wr_a:.0%})")

In [ ]:
# === Step 10 — Automated metrics (perplexity under base, BERTScore vs context) ===
# Perplexity needs the BASE distribution, not the latest-loaded adapter's
# distribution, so we wrap the whole call in `disable_adapter()`.
from eval.metrics import compute_all_auto_metrics

with model.disable_adapter():
    auto_metrics = compute_all_auto_metrics(
        base_model=model,
        tokenizer=tokenizer,
        config=CONFIG,
        final_generations=final_gens,
        prompt_records=prompts,
    )

print("\n=== Automated metrics per variant ===")
for name, m in sorted(auto_metrics.items()):
    print(
        f"  {name:30s}  perplexity={m['perplexity_mean']:7.2f}  "
        f"bertscore={m['bertscore_mean']:+.4f}  n={int(m['n'])}"
    )

## Step 12 / 13 — Figures and qualitative results

Three figure groups feed the report:

1. **Pairwise win rates** — base vs DPO vs KTO at each context length. The headline result.
2. **Training curves** — loss + reward margin (DPO) / utility (KTO) over time, loaded from the JSONL logger output. Demonstrates the trainings actually converged.
3. **Qualitative examples** — a handful of cherry-picked outputs from each variant, plus a failure case or two for honesty.

All figures save to `<drive_root>/figures/` as both PNG (for the notebook PDF) and PDF (for the LaTeX report).

In [ ]:
# === Figure 1: Pairwise win rates (base vs DPO vs KTO, each context length) ===
import os
import matplotlib.pyplot as plt
import numpy as np

FIGURES_DIR = os.path.join(CONFIG["drive_root"], "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

def _save_figure(fig, name: str) -> None:
    for ext in ("png", "pdf"):
        path = os.path.join(FIGURES_DIR, f"{name}.{ext}")
        fig.savefig(path, bbox_inches="tight", dpi=200)
    print(f"  saved figures/{name}.{{png,pdf}}")

def _winrate_versus_base(records, ctx, opponent_substr):
    """Win rate of opponent (e.g. 'dpo' or 'kto') vs base in one context."""
    wins = losses = 0
    for r in records:
        if r["context_length"] != ctx:
            continue
        ranks = r["variant_rankings"]
        opp = next((v for v in ranks if opponent_substr in v), None)
        if opp is None or "base" not in ranks:
            continue
        if ranks.index(opp) < ranks.index("base"):
            wins += 1
        else:
            losses += 1
    return wins / (wins + losses) if (wins + losses) else 0.0

# Compute win rates for the headline bar chart
algos = ["dpo", "kto"]
contexts = ["short", "long"]
winrates = np.array([
    [_winrate_versus_base(all_records, ctx, algo) for ctx in contexts]
    for algo in algos
])

fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(contexts))
width = 0.35
ax.bar(x - width / 2, winrates[0], width, label="DPO vs base", color="#4C78A8")
ax.bar(x + width / 2, winrates[1], width, label="KTO vs base", color="#F58518")
ax.axhline(0.5, color="black", linestyle=":", linewidth=1, label="parity")
ax.set_ylim(0, 1)
ax.set_xticks(x)
ax.set_xticklabels([c.title() + " context" for c in contexts])
ax.set_ylabel("Win rate vs. base Mistral")
ax.set_title("Trained variants vs. base Mistral (LLM cross-judge, n=30 prompts)")
ax.legend()
for i, ctx in enumerate(contexts):
    for j, algo in enumerate(algos):
        ax.text(
            x[i] + (j - 0.5) * width, winrates[j, i] + 0.02,
            f"{winrates[j, i]:.0%}",
            ha="center", fontsize=9,
        )
plt.tight_layout()
_save_figure(fig, "winrates_vs_base")
plt.show()

In [ ]:
# === Figure 2: Training loss curves for all 4 variants ===
import json

VARIANT_LABELS = {
    "dpo_rubric_short":  "DPO (short)",
    "dpo_rubric_long":   "DPO (long)",
    "kto_binary_short":  "KTO (short)",
    "kto_binary_long":   "KTO (long)",
}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax_train, ax_val = axes

for exp_name, label in VARIANT_LABELS.items():
    jsonl_path = os.path.join(
        CONFIG["drive_root"], "metrics", exp_name, "log.jsonl"
    )
    if not os.path.isfile(jsonl_path):
        print(f"  no log for {exp_name} (training may not have started)")
        continue
    rows = []
    with open(jsonl_path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    train_rows = [r for r in rows if r.get("phase") == "train"]
    val_rows   = [r for r in rows if r.get("phase") == "val"]
    if train_rows:
        ax_train.plot(
            [r["step"] for r in train_rows],
            [r["loss"]  for r in train_rows],
            label=label, linewidth=1.5,
        )
    if val_rows:
        ax_val.plot(
            [r["step"] for r in val_rows],
            [r["loss"]  for r in val_rows],
            label=label, marker="o", linewidth=1.5,
        )

ax_train.set_title("Training loss")
ax_train.set_xlabel("Optimizer step")
ax_train.set_ylabel("Loss")
ax_train.legend(fontsize=9)
ax_val.set_title("Validation loss")
ax_val.set_xlabel("Optimizer step")
ax_val.set_ylabel("Loss")
ax_val.legend(fontsize=9)
plt.tight_layout()
_save_figure(fig, "training_curves")
plt.show()

In [ ]:
# === Sample outputs: pick a handful of eval prompts and print all 5 variants ===
# Useful for the report's qualitative-results section. Eyeball, then cherry-pick.
import random

SAMPLE_RNG = random.Random(0)
SAMPLE_N = 4  # how many eval prompts to print

eval_prompts = [p for p in prompts if p["split"] == "eval"]
sampled = SAMPLE_RNG.sample(eval_prompts, min(SAMPLE_N, len(eval_prompts)))

for p in sampled:
    pid = p["prompt_id"]
    print("=" * 80)
    print(f"prompt_id: {pid}   style: {p['style']}")
    print(f"headline:  {p['short_context'].splitlines()[0][:90]}")
    print("=" * 80)
    for ctx in ("short", "long"):
        print(f"\n--- {ctx.upper()} CONTEXT ---")
        for variant in ("base", f"dpo_rubric_{ctx}", f"kto_binary_{ctx}"):
            resp = gens_by_prompt.get(pid, {}).get((variant, ctx))
            if resp is None:
                continue
            print(f"\n  [{variant}]")
            for line in resp.strip().splitlines():
                print(f"    {line}")
    print()